[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [1]:
# Install the latest torch-judge from this repo in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/CharlesShang/TorchCode.git@master')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.8 MB/s eta 0:00:00


In [10]:
import torch
import math
# torch.triu?
help(torch.masked_fill)

Help on built-in function masked_fill in module torch:

masked_fill(...)



In [33]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):
    # Q ~ [B, S, D]
    B, S, D = Q.shape
    QK = Q @ K.transpose(1, 2) / math.sqrt(D)
    mask = torch.triu(
        torch.ones(S, S, dtype=torch.bool, device=Q.device),
        diagonal=1,
    )
    QK.masked_fill_(mask, float("-inf"))
    # print(f"QK: {QK.shape}")

    return torch.softmax(QK, dim=-1) @ V

In [31]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

QK: torch.Size([1, 4, 4])
Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [32]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
QK: torch.Size([2, 6, 6])
  ✅ [1/4] Output shape (2.5ms)
QK: torch.Size([1, 8, 8])
QK: torch.Size([1, 8, 8])
  ✅ [2/4] Future tokens don't affect past (2.2ms)
QK: torch.Size([1, 4, 4])
  ✅ [3/4] First position only sees itself (1.4ms)
QK: torch.Size([2, 4, 4])
  ✅ [4/4] Gradient flow (0.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (6.8ms total)
  Progress saved. Run status() to see your dashboard.

